**Create BigQuery Table**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Users"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

schema = [

    bigquery.SchemaField(
        "User_ID",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Email",
        "STRING"
    ),

    bigquery.SchemaField(
        "Password_Hash",
        "STRING"
    ),

    bigquery.SchemaField(
        "Role",
        "STRING"
    ),

    bigquery.SchemaField(
        "Dr_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Is_Active",
        "STRING"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Role",
    "Dr_Code"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Users


**Create Postgres Table**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS user_no_seq;

""")

create_table_query = """
CREATE TABLE IF NOT EXISTS Users (

    User_ID TEXT PRIMARY KEY
    DEFAULT (
        'User' ||
        LPAD(
            nextval('user_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Email TEXT,

    Password_Hash TEXT,

    Role TEXT,

    Dr_Code TEXT,

    Is_Active TEXT

);
"""

cursor.execute(create_table_query)

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_user_email
ON Users(Email);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_user_role
ON Users(Role);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_user_dr_code
ON Users(Dr_Code);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Users")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Users


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

query = """

SELECT

    User_ID,
    Email,
    Password_Hash,
    Role,
    Dr_Code,
    Is_Active

FROM `depihealthnux.Depihealthnux_Main.Users`

ORDER BY User_ID

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Users;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Users (

            User_ID,
            Email,
            Password_Hash,
            Role,
            Dr_Code,
            Is_Active

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

cursor.execute("""

SELECT setval(
    'user_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(User_ID,'User','')
                    AS INTEGER
                )
            )
            FROM Users
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Users
ORDER BY User_ID
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [User_ID, Email, Password_Hash, Role, Dr_Code, Is_Active]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres to BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    User_ID,
    Email,
    Password_Hash,
    Role,
    Dr_Code,
    Is_Active

FROM Users

ORDER BY User_ID

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Users",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Users`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_1836\4001729372.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Empty DataFrame
Columns: [user_id, email, password_hash, role, dr_code, is_active]
Index: []
Rows Retrieved: 0
Uploaded 0 rows

First 3 Rows From BigQuery
Empty DataFrame
Columns: [user_id, email, password_hash, role, dr_code, is_active]
Index: []
